# Song Familiarity Dataset EDA

This notebook provides a first-pass analysis of the dataset stored in `ds005876/`, which is the canonical dataset directory kept in this repository after removing the redundant `data/` copy.

It covers:
- dataset documentation from the dataset `README` and JSON sidecars
- a simple inventory of available files
- participant-level descriptive statistics
- trial-level behavioral descriptive statistics
- basic exploratory data analysis (EDA)
- lightweight event-derived summaries from the EEG `events.tsv` files
- EEG metadata summaries and conditional signal-level plots when raw EEGLAB files are available


## Reading Guide

The notebook moves from file-level inventory to participant demographics, behavioral responses, song-level difficulty, event-derived timing features, and finally EEG metadata. Sections 1 through 5 rely only on BIDS tabular files, while Section 6 adds metadata and optional signal plots when the raw `.set` / `.fdt` binaries are available locally.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")

PALETTE = ["#16324F", "#2A9D8F", "#E9C46A", "#E76F51", "#6D597A", "#264653"]
sns.set_theme(style="whitegrid", context="notebook", palette=PALETTE)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10


def locate_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "data").exists() or (candidate / "ds005876").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")


def flatten_schema(schema: dict) -> pd.DataFrame:
    rows = []
    for field, meta in schema.items():
        levels = meta.get("Levels", {}) if isinstance(meta, dict) else {}
        level_text = ", ".join(f"{key}={value}" for key, value in levels.items()) if levels else ""
        rows.append(
            {
                "field": field,
                "description": meta.get("Description", "") if isinstance(meta, dict) else "",
                "units": meta.get("Units", "") if isinstance(meta, dict) else "",
                "levels": level_text,
            }
        )
    return pd.DataFrame(rows)


def load_all_behavioral(data_dir: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(data_dir.glob("sub-*/beh/*_beh.tsv")):
        df = pd.read_csv(path, sep="\t")
        df["participant_id"] = path.parent.parent.name
        frames.append(df)
    if not frames:
        raise FileNotFoundError("No behavioral TSV files were found.")
    behavior = pd.concat(frames, ignore_index=True)
    behavior["rt_numeric"] = pd.to_numeric(behavior["rt"], errors="coerce")
    behavior["promptRT_numeric"] = pd.to_numeric(behavior["promptRT"], errors="coerce")
    behavior["rtMC_numeric"] = pd.to_numeric(behavior["rtMC"], errors="coerce")
    return behavior


def summarize_event_file(events_df: pd.DataFrame, participant_id: str) -> dict:
    trial_starts = events_df[
        (events_df["value"].astype(str) == "1")
        & (events_df["stim_file"].astype(str) != "n/a")
    ]
    return {
        "participant_id": participant_id,
        "n_event_rows": len(events_df),
        "n_trial_starts": len(trial_starts),
        "n_note_onsets": int((events_df["value"].astype(str) == "noteOnset").sum()),
        "first_onset_sec": float(events_df["onset"].min()),
        "last_onset_sec": float(events_df["onset"].max()),
    }


def add_note_features(behavior: pd.DataFrame, data_dir: Path) -> pd.DataFrame:
    enriched = behavior.copy()
    enriched["note_count"] = np.nan
    enriched["note_rate"] = np.nan

    for participant_id in sorted(enriched["participant_id"].unique()):
        events_path = data_dir / participant_id / "eeg" / f"{participant_id}_task-songfamiliarity_events.tsv"
        events_df = pd.read_csv(events_path, sep="\t")
        trial_starts = events_df[
            (events_df["value"].astype(str) == "1")
            & (events_df["stim_file"].astype(str) != "n/a")
        ].reset_index(drop=True)

        participant_trials = enriched[enriched["participant_id"] == participant_id].sort_values("trialNum")
        for index in range(min(len(trial_starts), len(participant_trials))):
            start_onset = trial_starts.iloc[index]["onset"]
            end_onset = (
                trial_starts.iloc[index + 1]["onset"]
                if index + 1 < len(trial_starts)
                else events_df["onset"].iloc[-1] + 1
            )
            mask = (
                (events_df["value"].astype(str) == "noteOnset")
                & (events_df["onset"] >= start_onset)
                & (events_df["onset"] < end_onset)
            )
            note_count = int(mask.sum())
            trial_duration_sec = end_onset - start_onset
            row_index = participant_trials.index[index]
            enriched.at[row_index, "note_count"] = note_count
            enriched.at[row_index, "note_rate"] = note_count / trial_duration_sec if trial_duration_sec > 0 else np.nan

    return enriched


ROOT = locate_project_root()
DATA_DIR = ROOT / "ds005876" if (ROOT / "ds005876").exists() else ROOT / "data"
print(f"Project root: {ROOT}")
print(f"Dataset directory: {DATA_DIR}")


Project root: C:\Users\jnhdv\School Projects\COGS 189\COGS-189-Group-Project-WI26
Dataset directory: C:\Users\jnhdv\School Projects\COGS 189\COGS-189-Group-Project-WI26\ds005876


## 1. Dataset Documentation And Inventory

Start by reading the dataset `README` plus the JSON sidecars that describe the BIDS dataset and tabular fields.


In [2]:
dataset_readme = (DATA_DIR / "README").read_text(encoding="utf-8").strip()
dataset_description = json.loads((DATA_DIR / "dataset_description.json").read_text(encoding="utf-8"))
participants_schema = json.loads((DATA_DIR / "participants.json").read_text(encoding="utf-8"))
events_schema = json.loads((DATA_DIR / "task-songfamiliarity_events.json").read_text(encoding="utf-8"))

subject_dirs = sorted(path.name for path in DATA_DIR.glob("sub-*") if path.is_dir())
inventory = pd.DataFrame(
    {
        "artifact": [
            "subject folders",
            "behavioral TSV files",
            "event TSV files",
            "EEG .set files",
            "EEG .fdt files",
            "stimulus WAV files",
            "derivative audio files",
        ],
        "count": [
            len(subject_dirs),
            len(list(DATA_DIR.glob("sub-*/beh/*_beh.tsv"))),
            len(list(DATA_DIR.glob("sub-*/eeg/*_events.tsv"))),
            len(list(DATA_DIR.glob("sub-*/eeg/*.set"))),
            len(list(DATA_DIR.glob("sub-*/eeg/*.fdt"))),
            len(list(DATA_DIR.glob("stimuli/*.wav"))),
            len(list(DATA_DIR.glob("derivatives/sub-*/audio/*.wav"))),
        ],
    }
)

example_event_columns = pd.read_csv(
    next(DATA_DIR.glob("sub-*/eeg/*_events.tsv")),
    sep="\t",
    nrows=0,
).columns.tolist()

print("Dataset README\n")
print(dataset_readme)
print("\nExample events.tsv columns:", example_event_columns)

display(inventory)
display(pd.Series(dataset_description, name="value").to_frame())
display(flatten_schema(participants_schema))
display(flatten_schema(events_schema))


Dataset README

# Song Familiarity

Twenty-nine participants listened to song melodies and responded as soon as the song felt familiar. Participants were then asked to identify the song, if possible (title, artist, or lyrics). Next, participants were shown a multiple choice display with four song titles, selected a song title, and were given visual feedback (correct: selected option turned green and a checkmark appeared next to the title; incorrect: selected option turned red and an x appeared next to the title.)

Song stimuli are taken from Kostic and Cleary (2009): https://supp.apa.org/psycarticles/supplemental/a0014584/a0014584_supp.html

An audio file with a reconstruction of what each participant heard throughout the experiment can be found in /derivatives. The audio file has been synchronized with the EEG recording.

Example events.tsv columns: ['onset', 'duration', 'sample', 'value', 'stim_file']


,artifact,count
0,subject folders,29
1,behavioral TSV files,29
2,event TSV files,29
3,EEG .set files,29
4,EEG .fdt files,29
5,stimulus WAV files,121
6,derivative audio files,0


,value
Name,Song Familiarity
Authors,"[Jared R. Girard, Aaron M. Bishop, Cameron D. ..."
License,CC0
DatasetDOI,doi:10.18112/openneuro.ds005876.v1.0.1
ReferencesAndLinks,[]
BIDSVersion,1.8.0
HEDVersion,8.1.0
GeneratedBy,"[{'Name': 'bids-matlab-tools', 'Version': '9.0'}]"


,field,description,units,levels
0,participant_id,participant number,,
1,datetime,date and time at start of task,,
2,age,self-reported age of participant,,
3,sex,self-reported sex of participant,,"M=male, F=female"
4,cch,coarse and curly hair type,,"Y=yes, N=no"
5,handedness,self-reported handedness of participant,,"L=left-handed, R=right-handed, LR=ambidextrous"


,field,description,units,levels
0,type,Event value,,"x1=Start of trial, x2=Response (spacebar), x3=..."
1,latency,Event onset,samples,
2,urevent,Event number,,
3,duration,Event duration,samples,
4,stim_file,Stimulus file,,


## 2. Participant-Level Description

Load `participants.tsv`, parse the session timestamp, and summarize the demographic fields described in `participants.json`.


In [3]:
participants = pd.read_csv(DATA_DIR / "participants.tsv", sep="\t")
participants["datetime"] = pd.to_datetime(participants["datetime"], format="%d-%b-%Y %H:%M:%S")
participants["session_date"] = participants["datetime"].dt.date

participant_overview = pd.DataFrame(
    {
        "metric": ["n_participants", "mean_age", "median_age", "min_age", "max_age"],
        "value": [
            len(participants),
            round(participants["age"].mean(), 2),
            round(participants["age"].median(), 2),
            int(participants["age"].min()),
            int(participants["age"].max()),
        ],
    }
)

categorical_counts = pd.concat(
    [
        participants["sex"].value_counts(dropna=False).rename("count").rename_axis("category").reset_index().assign(field="sex"),
        participants["handedness"].value_counts(dropna=False).rename("count").rename_axis("category").reset_index().assign(field="handedness"),
        participants["cch"].value_counts(dropna=False).rename("count").rename_axis("category").reset_index().assign(field="cch"),
    ],
    ignore_index=True,
)[["field", "category", "count"]]

display(participants.head())
display(participant_overview)
display(participants[["age"]].describe().round(2).T)
display(categorical_counts)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(participants, x="age", bins=10, color=PALETTE[0], edgecolor="white", ax=axes[0])
axes[0].set_title("Age distribution")
axes[0].set_xlabel("Age")

sns.countplot(data=participants, x="sex", order=participants["sex"].value_counts().index, color=PALETTE[1], ax=axes[1])
axes[1].set_title("Sex")
axes[1].set_xlabel("Sex")

sns.countplot(
    data=participants,
    x="handedness",
    order=participants["handedness"].value_counts().index,
    color=PALETTE[4],
    ax=axes[2],
)
axes[2].set_title("Handedness")
axes[2].set_xlabel("Handedness")

plt.tight_layout()
plt.show()


,participant_id,datetime,age,sex,handedness,cch,session_date
0,sub-01,2024-04-03 12:49:04,24,M,L,N,2024-04-03
1,sub-02,2024-04-26 10:02:41,27,M,R,N,2024-04-26
2,sub-03,2024-05-02 10:30:25,44,M,R,N,2024-05-02
3,sub-04,2024-05-13 10:20:51,23,F,R,N,2024-05-13
4,sub-05,2024-05-14 10:31:23,18,F,R,N,2024-05-14


,metric,value
0,n_participants,29.00
1,mean_age,22.76
2,median_age,23.00
3,min_age,18.00
4,max_age,44.00


,count,mean,std,min,25%,50%,75%,max
age,29.0,22.76,5.43,18.0,18.0,23.0,24.0,44.0


,field,category,count
0,sex,F,19
1,sex,M,10
2,handedness,R,25
3,handedness,L,3
4,handedness,LR,1
5,cch,N,25
6,cch,Y,4


### Participant Interpretation

The sample contains `29` participants with mean age `22.76` years and an age range of `18` to `44`. That is enough for descriptive EDA, but it is still a modest sample for EEG generalization claims, so later participant-aware model evaluation is a methodological necessity rather than a nice-to-have.


## 3. Behavioral Trials

Each subject has a behavioral TSV with one row per trial. Here we concatenate those files, coerce reaction-time fields to numeric values, and inspect the main behavioral variables.


In [4]:
behavior = load_all_behavioral(DATA_DIR)

behavior_overview = pd.DataFrame(
    {
        "metric": [
            "n_trials",
            "n_participants",
            "unique_songs_presented",
            "familiarity_response_rate",
            "multiple_choice_accuracy",
            "valid_familiarity_rt_count",
        ],
        "value": [
            len(behavior),
            behavior["participant_id"].nunique(),
            behavior["songFileName"].nunique(),
            round(behavior["responded"].mean(), 4),
            round(behavior["outcomeMC"].mean(), 4),
            int(behavior["rt_numeric"].notna().sum()),
        ],
    }
)

numeric_columns = [
    "songDur",
    "fixTime",
    "postMusicTime",
    "rt_numeric",
    "promptRT_numeric",
    "rtMC_numeric",
    "preFeedbackTime",
]

numeric_summary = behavior[numeric_columns].describe().round(3).T

missingness = pd.DataFrame(
    {
        "missing_fraction": behavior[["rt", "reply", "promptRT", "respMC", "rtMC"]]
        .replace("n/a", np.nan)
        .isna()
        .mean()
        .round(3)
    }
).sort_values("missing_fraction", ascending=False)

display(behavior.head())
display(behavior_overview)
display(numeric_summary)
display(missingness)


,trialNum,songNumber,songFileName,songDur,fixTime,responded,rt,postMusicTime,reply,promptRT,choicesMC,respMC,rtMC,outcomeMC,preFeedbackTime,participant_id,rt_numeric,promptRT_numeric,rtMC_numeric
0,1,34,do wah ditty.wav,7.758005,0.457367,0,NaN,0.491517,NaN,11.723581,7 34 31 12,2,5.467003,1,1.141405,sub-01,NaN,11.723581,5.467003
1,2,11,alouette.wav,5.946984,0.450400,1,2.872449,0.462234,Frere Jaque,8.877756,97 61 11 29,3,2.955956,1,1.043290,sub-01,2.872449,8.877756,2.955956
2,3,45,great escape.wav,15.750023,0.458773,1,10.306871,0.488652,EEG Lab,7.080602,101 16 6 45,2,3.429529,0,1.197642,sub-01,10.306871,7.080602,3.429529
3,4,24,buffalo gals.wav,9.331020,0.472790,1,3.721228,0.481305,Childhood lullaby,7.650939,24 61 55 58,4,3.680033,0,1.027963,sub-01,3.721228,7.650939,3.680033
4,5,85,over the river and through the woods.wav,13.913016,0.452219,0,NaN,0.402142,NaN,30.119903,35 66 85 64,1,3.465673,0,1.001730,sub-01,NaN,30.119903,3.465673


,metric,value
0,n_trials,2074.0000
1,n_participants,29.0000
2,unique_songs_presented,121.0000
3,familiarity_response_rate,0.4966
4,multiple_choice_accuracy,0.5535
5,valid_familiarity_rt_count,1030.0000


,count,mean,std,min,25%,50%,75%,max
songDur,2074.0,9.547,2.289,4.778,8.063,9.311,10.602,17.067
fixTime,2074.0,0.449,0.029,0.400,0.425,0.449,0.474,0.500
postMusicTime,2074.0,0.451,0.028,0.400,0.427,0.450,0.475,0.500
rt_numeric,1030.0,4.497,2.217,1.010,2.834,4.057,5.776,14.596
promptRT_numeric,2074.0,4.771,5.187,0.297,1.465,2.739,6.385,48.774
rtMC_numeric,2074.0,5.137,3.328,0.371,2.797,4.338,6.580,33.667
preFeedbackTime,2074.0,1.099,0.057,1.000,1.051,1.096,1.147,1.200


,missing_fraction
reply,0.674
rt,0.503
promptRT,0.000
respMC,0.000
rtMC,0.000


In [5]:
subject_level = behavior.groupby("participant_id").agg(
    n_trials=("trialNum", "size"),
    response_rate=("responded", "mean"),
    mc_accuracy=("outcomeMC", "mean"),
    median_rt=("rt_numeric", "median"),
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ordered_response = subject_level.sort_values("response_rate")
sns.barplot(data=ordered_response, y="participant_id", x="response_rate", color=PALETTE[1], ax=axes[0, 0])
axes[0, 0].set_title("Familiarity response rate by participant")
axes[0, 0].set_xlabel("Response rate")
axes[0, 0].set_ylabel("Participant")

sns.histplot(behavior.dropna(subset=["rt_numeric"]), x="rt_numeric", bins=30, color=PALETTE[0], edgecolor="white", ax=axes[0, 1])
axes[0, 1].axvline(behavior["rt_numeric"].median(), color=PALETTE[3], linestyle="--")
axes[0, 1].set_title("Familiarity RT distribution")
axes[0, 1].set_xlabel("RT (seconds)")

ordered_accuracy = subject_level.sort_values("mc_accuracy")
sns.barplot(data=ordered_accuracy, y="participant_id", x="mc_accuracy", color=PALETTE[4], ax=axes[1, 0])
axes[1, 0].set_title("Multiple-choice accuracy by participant")
axes[1, 0].set_xlabel("Accuracy")
axes[1, 0].set_ylabel("Participant")

song_duration_plot = behavior.copy()
song_duration_plot["responded_label"] = song_duration_plot["responded"].map({0: "No", 1: "Yes"})
sns.boxplot(data=song_duration_plot, x="responded_label", y="songDur", palette=[PALETTE[2], PALETTE[1]], ax=axes[1, 1])
axes[1, 1].set_title("Song duration by familiarity response")
axes[1, 1].set_xlabel("Responded as familiar")
axes[1, 1].set_ylabel("Song duration (seconds)")

plt.tight_layout()
plt.show()

display(subject_level.describe(include="all").round(3))


,participant_id,n_trials,response_rate,mc_accuracy,median_rt
count,29,29.000,29.000,29.000,28.000
unique,29,NaN,NaN,NaN,NaN
top,sub-01,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN
mean,NaN,71.517,0.486,0.548,4.302
std,NaN,11.243,0.224,0.170,0.944
min,NaN,51.000,0.000,0.162,2.335
25%,NaN,63.000,0.333,0.446,3.827
50%,NaN,69.000,0.524,0.544,4.461
75%,NaN,77.000,0.627,0.667,4.823


### Behavioral Interpretation

Across `2,074` total trials, participants marked songs as familiar on `49.7%` of trials and answered the follow-up multiple-choice identification item correctly on `55.4%` of trials. The RT distributions and participant bars show real heterogeneity rather than a trivial task, which is useful context for both the analysis-ready feature building and the later EEG modeling notebooks.


## 4. Song-Level Patterns

Aggregating across participants can highlight songs that are recognized quickly or often, versus songs that are rarely reported as familiar.


In [6]:
song_summary = behavior.groupby("songFileName").agg(
    n_trials=("songFileName", "size"),
    response_rate=("responded", "mean"),
    mc_accuracy=("outcomeMC", "mean"),
    mean_rt=("rt_numeric", "mean"),
    median_rt=("rt_numeric", "median"),
).reset_index()

frequent_songs = song_summary.query("n_trials >= 10").copy()

display(frequent_songs.sort_values("response_rate", ascending=False).head(10).round(3))
display(frequent_songs.sort_values("response_rate", ascending=True).head(10).round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top_songs = frequent_songs.sort_values("response_rate", ascending=False).head(12).sort_values("response_rate")
bottom_songs = frequent_songs.sort_values("response_rate", ascending=True).head(12).sort_values("response_rate")

sns.barplot(data=top_songs, y="songFileName", x="response_rate", color=PALETTE[1], ax=axes[0])
axes[0].set_title("Most familiar songs in the sample")
axes[0].set_xlabel("Response rate")
axes[0].set_ylabel("Song")

sns.barplot(data=bottom_songs, y="songFileName", x="response_rate", color=PALETTE[3], ax=axes[1])
axes[1].set_title("Least familiar songs in the sample")
axes[1].set_xlabel("Response rate")
axes[1].set_ylabel("Song")

plt.tight_layout()
plt.show()


,songFileName,n_trials,response_rate,mc_accuracy,mean_rt,median_rt
11,amazing grace.wav,14,1.000,1.000,4.569,4.479
45,happy birthday.wav,17,1.000,0.882,2.739,2.873
81,old mcdonald.wav,16,1.000,0.875,2.943,2.384
94,row your boat.wav,17,1.000,0.941,3.438,2.743
115,this old man.wav,20,0.950,0.500,4.164,3.952
68,mary had a little lamb.wav,18,0.944,0.889,2.985,2.543
13,angel is my centerfold (Example).wav,14,0.929,0.429,3.755,3.713
15,are you sleeping brother john.wav,14,0.929,0.643,3.961,3.260
34,do your ears hang low.wav,20,0.900,0.650,3.305,3.262
27,chicken dance.wav,17,0.882,0.765,2.786,2.789


,songFileName,n_trials,response_rate,mc_accuracy,mean_rt,median_rt
38,everlasting love.wav,19,0.053,0.316,2.070,2.070
46,happy trails.wav,18,0.056,0.444,6.312,6.312
9,all my loving.wav,19,0.105,0.474,3.920,3.920
37,eight days a week.wav,19,0.105,0.474,3.098,3.098
18,billie jean.wav,18,0.111,0.500,4.051,4.051
60,let it be.wav,16,0.125,0.312,6.385,6.385
64,louie louie.wav,16,0.125,0.312,3.943,3.943
32,do as I'm doing.wav,15,0.133,0.467,3.342,3.342
111,the brady bunch.wav,14,0.143,0.286,6.533,6.533
86,peter and the wolf.wav,19,0.158,0.316,3.044,2.898


### Song-Level Interpretation

Song familiarity is highly uneven. Among songs observed at least 10 times, `amazing grace.wav` is familiar on every observed trial (`14/14`), while `everlasting love.wav` is familiar on only `5.3%` of its `19` trials. That spread matters because part of the prediction problem may come from stable song-level base rates rather than purely participant- or EEG-level signal.


## 5. Event-Derived Summaries From EEG Event Logs

The notebook does not load the raw EEG signal files (`.set` / `.fdt`), but it does use the per-subject `events.tsv` tables to summarize trial structure and derive simple note-onset features.

One thing to notice: the JSON sidecar uses labels like `type`, `latency`, and `urevent`, while the actual BIDS event TSV exposes columns such as `onset`, `duration`, `sample`, `value`, and `stim_file`.


In [7]:
event_summaries = []
event_value_counts = []
for events_path in sorted(DATA_DIR.glob("sub-*/eeg/*_events.tsv")):
    events_df = pd.read_csv(events_path, sep="\t")
    participant_id = events_path.parent.parent.name
    event_summaries.append(summarize_event_file(events_df, participant_id))

    value_counts = events_df["value"].astype(str).value_counts().rename_axis("value").reset_index(name="count")
    value_counts["participant_id"] = participant_id
    event_value_counts.append(value_counts)

event_summary = pd.DataFrame(event_summaries)
event_value_counts = pd.concat(event_value_counts, ignore_index=True)
overall_event_counts = event_value_counts.groupby("value")["count"].sum().sort_values(ascending=False).to_frame()

behavior_with_events = add_note_features(behavior, DATA_DIR)
responded_with_events = behavior_with_events.dropna(subset=["rt_numeric", "note_rate"]).copy()

event_alignment = behavior_with_events.groupby("participant_id").size().reset_index(name="behavior_trials").merge(
    event_summary[["participant_id", "n_trial_starts"]],
    on="participant_id",
    how="left",
)
event_alignment["trial_count_match"] = event_alignment["behavior_trials"] == event_alignment["n_trial_starts"]

display(event_summary.describe().round(2).T)
display(overall_event_counts)
display(event_alignment)
display(behavior_with_events[["note_count", "note_rate"]].describe().round(3).T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(behavior_with_events, x="note_rate", bins=30, color=PALETTE[4], edgecolor="white", ax=axes[0])
axes[0].set_title("Distribution of note onset rate")
axes[0].set_xlabel("Note onsets per second")

sns.regplot(
    data=responded_with_events,
    x="note_rate",
    y="rt_numeric",
    scatter_kws={"alpha": 0.25, "s": 24, "color": PALETTE[0]},
    line_kws={"color": PALETTE[3]},
    ax=axes[1],
)
axes[1].set_title("Note rate vs familiarity RT")
axes[1].set_xlabel("Note onsets per second")
axes[1].set_ylabel("RT (seconds)")

plt.tight_layout()
plt.show()

print(f"Correlation between note_rate and familiarity RT: {responded_with_events['note_rate'].corr(responded_with_events['rt_numeric']):.3f}")
print(f"Correlation between song duration and familiarity RT: {responded_with_events['songDur'].corr(responded_with_events['rt_numeric']):.3f}")


,count,mean,std,min,25%,50%,75%,max
n_event_rows,29.0,3519.72,612.52,2623.00,2956.00,3459.00,3896.00,4985.00
n_trial_starts,29.0,71.52,11.24,51.00,63.00,69.00,77.00,101.00
n_note_onsets,29.0,3055.10,603.37,2180.00,2564.00,2978.00,3402.00,4452.00
first_onset_sec,29.0,179.23,76.17,15.67,145.24,198.65,230.31,285.77
last_onset_sec,29.0,1979.81,75.65,1793.46,1939.37,2005.38,2028.80,2091.26


,count
value,
noteOnset,88598
1,2074
3,2074
5,2074
4,2074
6,2074
8,1148
2,1030
7,926


,participant_id,behavior_trials,n_trial_starts,trial_count_match
0,sub-01,74,74,True
1,sub-02,75,75,True
2,sub-03,101,101,True
3,sub-04,78,78,True
4,sub-05,66,66,True
5,sub-06,72,72,True
6,sub-07,79,79,True
7,sub-09,61,61,True
8,sub-10,63,63,True
9,sub-11,59,59,True


,count,mean,std,min,25%,50%,75%,max
note_count,2074.0,42.718,22.032,6.000,26.000,40.000,54.000,128.000
note_rate,2074.0,1.751,0.868,0.208,1.114,1.624,2.251,5.525


Correlation between note_rate and familiarity RT: 0.582
Correlation between song duration and familiarity RT: 0.279


In [8]:
top_song = frequent_songs.sort_values("response_rate", ascending=False).iloc[0]
bottom_song = frequent_songs.sort_values("response_rate", ascending=True).iloc[0]
takeaways = [
    f"Participants: {len(participants)} total; mean age {participants['age'].mean():.2f} years (range {participants['age'].min()}-{participants['age'].max()}).",
    f"Behavioral trials: {len(behavior)} total across {behavior['participant_id'].nunique()} participants and {behavior['songFileName'].nunique()} unique songs.",
    f"Familiarity responses occurred on {behavior['responded'].mean():.1%} of trials; multiple-choice accuracy was {behavior['outcomeMC'].mean():.1%}.",
    f"Among responded trials, median familiarity RT was {behavior['rt_numeric'].median():.2f} seconds.",
    f"Most familiar high-coverage song: {top_song['songFileName']} (response rate {top_song['response_rate']:.1%}, n={int(top_song['n_trials'])}).",
    f"Least familiar high-coverage song: {bottom_song['songFileName']} (response rate {bottom_song['response_rate']:.1%}, n={int(bottom_song['n_trials'])}).",
    f"Event files aligned cleanly with the behavioral tables for all participants: {event_alignment['trial_count_match'].all()}.",
    f"Trial note rate showed a moderate positive association with familiarity RT (r = {responded_with_events['note_rate'].corr(responded_with_events['rt_numeric']):.3f}).",
]

print("Key takeaways\n")
for item in takeaways:
    print(f"- {item}")


Key takeaways

- Participants: 29 total; mean age 22.76 years (range 18-44).
- Behavioral trials: 2074 total across 29 participants and 121 unique songs.
- Familiarity responses occurred on 49.7% of trials; multiple-choice accuracy was 55.4%.
- Among responded trials, median familiarity RT was 4.06 seconds.
- Most familiar high-coverage song: amazing grace.wav (response rate 100.0%, n=14).
- Least familiar high-coverage song: everlasting love.wav (response rate 5.3%, n=19).
- Event files aligned cleanly with the behavioral tables for all participants: True.
- Trial note rate showed a moderate positive association with familiarity RT (r = 0.582).


### Event-Level Interpretation

The behavioral and event logs align cleanly, so the note-onset features are structurally reliable for downstream analysis. The positive note-rate versus RT relationship suggests that denser melodies tend to be answered more slowly, which is a plausible task-complexity effect and a useful non-neural covariate to carry into later models.


## 6. EEG Metadata And Signal-Level EDA

This section extends the dataset summary from event logs to the actual EEG recording metadata in each `*_eeg.json` file.

It also includes signal-level plotting code for:
- participant-averaged, stimulus-onset-locked EEG traces across the shared channel set
- representative channel grand averages across participants
- a channel-by-time heatmap of the familiar vs not-familiar difference wave

These plots run only when the raw `.set` and `.fdt` binaries are available locally.


In [9]:
def is_annex_placeholder(path: Path) -> bool:
    head = path.read_bytes()[:256]
    text = head.decode("utf-8", errors="ignore")
    return ".git/annex/objects" in text or "git-lfs" in text or text.startswith("../") or text.startswith("../../")


eeg_metadata_rows = []
for eeg_json_path in sorted(DATA_DIR.glob("sub-*/eeg/*_eeg.json")):
    meta = json.loads(eeg_json_path.read_text(encoding="utf-8"))
    eeg_metadata_rows.append(
        {
            "participant_id": eeg_json_path.parent.parent.name,
            "eeg_channel_count": meta.get("EEGChannelCount"),
            "sampling_frequency_hz": meta.get("SamplingFrequency"),
            "recording_duration_sec": meta.get("RecordingDuration"),
            "eeg_reference": meta.get("EEGReference"),
            "recording_type": meta.get("RecordingType"),
            "manufacturer_model": meta.get("ManufacturersModelName"),
            "power_line_frequency_hz": meta.get("PowerLineFrequency"),
        }
    )

eeg_metadata = pd.DataFrame(eeg_metadata_rows)
set_files = sorted(DATA_DIR.glob("sub-*/eeg/*.set"))
fdt_files = sorted(DATA_DIR.glob("sub-*/eeg/*.fdt"))

eeg_file_status = pd.DataFrame(
    {
        "file_type": [".set", ".fdt"],
        "n_files": [len(set_files), len(fdt_files)],
        "n_annex_placeholders": [sum(is_annex_placeholder(path) for path in set_files), sum(is_annex_placeholder(path) for path in fdt_files)],
    }
)
eeg_file_status["n_real_files"] = eeg_file_status["n_files"] - eeg_file_status["n_annex_placeholders"]

eeg_metadata_summary = eeg_metadata[["eeg_channel_count", "sampling_frequency_hz", "recording_duration_sec", "power_line_frequency_hz"]].describe().round(2).T
eeg_categorical_summary = pd.concat(
    [
        eeg_metadata["eeg_reference"].value_counts(dropna=False).rename("count").rename_axis("category").reset_index().assign(field="eeg_reference"),
        eeg_metadata["recording_type"].value_counts(dropna=False).rename("count").rename_axis("category").reset_index().assign(field="recording_type"),
        eeg_metadata["manufacturer_model"].value_counts(dropna=False).rename("count").rename_axis("category").reset_index().assign(field="manufacturer_model"),
    ],
    ignore_index=True,
)[["field", "category", "count"]]

display(eeg_metadata.head())
display(eeg_metadata_summary)
display(eeg_categorical_summary)
display(eeg_file_status)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(eeg_metadata, x="recording_duration_sec", bins=12, color=PALETTE[5], edgecolor="white", ax=axes[0])
axes[0].set_title("EEG recording duration")
axes[0].set_xlabel("Duration (seconds)")

recording_vs_trials = eeg_metadata.merge(subject_level[["participant_id", "n_trials", "response_rate"]], on="participant_id", how="left")
sns.scatterplot(data=recording_vs_trials, x="recording_duration_sec", y="n_trials", hue="response_rate", palette="viridis", s=70, ax=axes[1])
axes[1].set_title("Recording duration vs behavioral trial count")
axes[1].set_xlabel("Recording duration (seconds)")
axes[1].set_ylabel("Behavioral trials")

plt.tight_layout()
plt.show()

REAL_EEG_SUBJECTS = [path.parent.parent.name for path in set_files if not is_annex_placeholder(path)]
print(f"Subjects with locally available raw EEGLAB data: {len(REAL_EEG_SUBJECTS)}")
if not REAL_EEG_SUBJECTS:
    print("All .set/.fdt files in this workspace are git-annex pointers. Metadata and event summaries are available, but signal-level EEG plots require the real binary files.")


,participant_id,eeg_channel_count,sampling_frequency_hz,recording_duration_sec,eeg_reference,recording_type,manufacturer_model,power_line_frequency_hz
0,sub-01,32,1000,1807.10,common,continuous,actiCHamp,60
1,sub-02,32,1000,2036.55,common,continuous,actiCHamp,60
2,sub-03,32,1000,1934.15,common,continuous,actiCHamp,60
3,sub-04,32,1000,1877.15,common,continuous,actiCHamp,60
4,sub-05,32,1000,1946.65,common,continuous,actiCHamp,60


,count,mean,std,min,25%,50%,75%,max
eeg_channel_count,29.0,32.00,0.00,32.0,32.00,32.0,32.00,32.00
sampling_frequency_hz,29.0,1000.00,0.00,1000.0,1000.00,1000.0,1000.00,1000.00
recording_duration_sec,29.0,1988.34,73.11,1807.1,1946.65,2011.1,2036.55,2097.15
power_line_frequency_hz,29.0,60.00,0.00,60.0,60.00,60.0,60.00,60.00


,field,category,count
0,eeg_reference,common,29
1,recording_type,continuous,29
2,manufacturer_model,actiCHamp,29


,file_type,n_files,n_annex_placeholders,n_real_files
0,.set,29,0,29
1,.fdt,29,0,29


Subjects with locally available raw EEGLAB data: 29


In [10]:
EEG_PLOTS_READY = False

if REAL_EEG_SUBJECTS:
    import mne

    def load_subject_stimulus_evokeds(participant_id, channel_order):
        set_path = DATA_DIR / participant_id / "eeg" / f"{participant_id}_task-songfamiliarity_eeg.set"
        events_path = DATA_DIR / participant_id / "eeg" / f"{participant_id}_task-songfamiliarity_events.tsv"
        behavior_path = DATA_DIR / participant_id / "beh" / f"{participant_id}_task-songfamiliarity_beh.tsv"

        raw = mne.io.read_raw_eeglab(set_path, preload=False, verbose=False)
        eeg_picks = mne.pick_types(raw.info, eeg=True, exclude="bads")
        events_subject = pd.read_csv(events_path, sep="	")
        behavior_subject = pd.read_csv(behavior_path, sep="	").sort_values("trialNum").reset_index(drop=True)

        trial_starts = events_subject[
            (events_subject["value"].astype(str) == "1")
            & (events_subject["stim_file"].astype(str) != "n/a")
        ].reset_index(drop=True)
        n_trials = min(len(trial_starts), len(behavior_subject))
        if n_trials == 0:
            return {}, None

        responded_codes = pd.to_numeric(behavior_subject["responded"], errors="coerce").fillna(0).to_numpy()[:n_trials]
        stimulus_events = np.column_stack(
            [
                (trial_starts["onset"].to_numpy()[:n_trials] * raw.info["sfreq"]).astype(int),
                np.zeros(n_trials, dtype=int),
                np.where(responded_codes == 1, 1, 2),
            ]
        )

        stimulus_epochs = mne.Epochs(
            raw,
            stimulus_events,
            event_id={"familiar": 1, "not_familiar": 2},
            tmin=-0.2,
            tmax=0.8,
            baseline=(None, 0),
            picks=eeg_picks,
            on_missing="ignore",
            preload=True,
            verbose=False,
        )

        evokeds = {}
        for label in ["familiar", "not_familiar"]:
            if len(stimulus_epochs[label]):
                evokeds[label] = stimulus_epochs[label].average().copy().pick(channel_order)

        trial_counts = {
            "participant_id": participant_id,
            "n_trials_total": n_trials,
            "n_familiar_epochs": len(stimulus_epochs["familiar"]),
            "n_not_familiar_epochs": len(stimulus_epochs["not_familiar"]),
        }

        del raw, stimulus_epochs
        return evokeds, trial_counts

    channel_lists = []
    for participant_id in REAL_EEG_SUBJECTS:
        set_path = DATA_DIR / participant_id / "eeg" / f"{participant_id}_task-songfamiliarity_eeg.set"
        raw = mne.io.read_raw_eeglab(set_path, preload=False, verbose=False)
        eeg_picks = mne.pick_types(raw.info, eeg=True, exclude="bads")
        channel_lists.append([raw.ch_names[index] for index in eeg_picks])

    common_channels = [
        channel for channel in channel_lists[0]
        if all(channel in channel_list for channel_list in channel_lists[1:])
    ]
    if not common_channels:
        raise RuntimeError("No shared EEG channels were found across participants with local data.")

    selected_channels = [channel for channel in ["Fz", "Cz", "Pz", "Oz"] if channel in common_channels]
    if not selected_channels:
        selected_channels = common_channels[: min(4, len(common_channels))]

    evoked_by_condition = {"familiar": [], "not_familiar": []}
    participant_mean_rows = []
    participant_trial_rows = []

    for participant_id in REAL_EEG_SUBJECTS:
        evokeds, trial_counts = load_subject_stimulus_evokeds(participant_id, common_channels)
        if trial_counts is not None:
            participant_trial_rows.append(trial_counts)

        for condition, evoked in evokeds.items():
            evoked_by_condition[condition].append(evoked)
            participant_mean_rows.append(
                pd.DataFrame(
                    {
                        "participant_id": participant_id,
                        "condition": condition,
                        "time_sec": evoked.times,
                        "amplitude_uv": evoked.data.mean(axis=0) * 1e6,
                    }
                )
            )

    participant_trial_summary = pd.DataFrame(participant_trial_rows).sort_values("participant_id").reset_index(drop=True)
    participant_trial_summary["familiar_fraction"] = (
        participant_trial_summary["n_familiar_epochs"] / participant_trial_summary["n_trials_total"]
    ).round(3)
    participant_mean_df = pd.concat(participant_mean_rows, ignore_index=True)

    grand_evokeds = {}
    for condition, evokeds in evoked_by_condition.items():
        if evokeds:
            grand_evokeds[condition] = mne.grand_average(evokeds, interpolate_bads=False, drop_bads=False)

    display(participant_trial_summary.head())
    display(participant_trial_summary[["n_trials_total", "n_familiar_epochs", "n_not_familiar_epochs", "familiar_fraction"]].describe().round(2).T)

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
    condition_colors = {"familiar": PALETTE[1], "not_familiar": PALETTE[3]}

    for axis, condition in zip(axes, ["familiar", "not_familiar"]):
        condition_df = participant_mean_df[participant_mean_df["condition"] == condition]
        if condition_df.empty:
            axis.text(0.5, 0.5, f"No {condition.replace('_', ' ')} epochs available", ha="center", va="center")
            axis.set_title(condition.replace("_", " ").title())
            continue

        for _, participant_df in condition_df.groupby("participant_id"):
            axis.plot(participant_df["time_sec"], participant_df["amplitude_uv"], color=condition_colors[condition], alpha=0.15, linewidth=0.8)

        grand_trace = grand_evokeds[condition].data.mean(axis=0) * 1e6
        axis.plot(grand_evokeds[condition].times, grand_trace, color=condition_colors[condition], linewidth=2.5)
        axis.axvline(0, color="black", linestyle="--", linewidth=1)
        axis.axhline(0, color="black", linestyle=":", linewidth=0.8)
        axis.set_title(f"{condition.replace('_', ' ').title()} trials\nmean across {len(common_channels)} channels")
        axis.set_xlabel("Time from stimulus onset (s)")

    axes[0].set_ylabel("Amplitude (uV)")
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(len(selected_channels), 1, figsize=(10, 2.8 * len(selected_channels)), sharex=True, sharey=True)
    if len(selected_channels) == 1:
        axes = [axes]

    for axis, channel in zip(axes, selected_channels):
        for condition in ["familiar", "not_familiar"]:
            if condition not in grand_evokeds:
                continue
            trace = grand_evokeds[condition].copy().pick([channel]).data[0] * 1e6
            axis.plot(
                grand_evokeds[condition].times,
                trace,
                color=condition_colors[condition],
                linewidth=2,
                label=condition.replace("_", " ").title(),
            )

        axis.axvline(0, color="black", linestyle="--", linewidth=1)
        axis.axhline(0, color="black", linestyle=":", linewidth=0.8)
        axis.set_title(f"Grand-average EEG at {channel}")
        axis.set_ylabel("Amplitude (uV)")
        axis.legend(loc="upper right")

    axes[-1].set_xlabel("Time from stimulus onset (s)")
    plt.tight_layout()
    plt.show()

    if {"familiar", "not_familiar"}.issubset(grand_evokeds):
        difference_evoked = mne.combine_evoked(
            [grand_evokeds["familiar"], grand_evokeds["not_familiar"]],
            weights=[1, -1],
        )
        downsample_step = 10
        difference_df = pd.DataFrame(
            difference_evoked.data[:, ::downsample_step] * 1e6,
            index=difference_evoked.ch_names,
            columns=np.round(difference_evoked.times[::downsample_step], 3),
        )

        plt.figure(figsize=(14, max(6, len(common_channels) * 0.22)))
        sns.heatmap(difference_df, cmap="RdBu_r", center=0, cbar_kws={"label": "Amplitude (uV)"})
        plt.title("Grand-average difference wave across channels (familiar - not familiar)")
        plt.xlabel("Time from stimulus onset (s)")
        plt.ylabel("Channel")
        plt.tight_layout()
        plt.show()

    EEG_PLOTS_READY = True
    print(
        f"Rendered participant-average EEG plots for {len(REAL_EEG_SUBJECTS)} participants "
        f"across {len(common_channels)} shared channels. Displayed channel traces: {selected_channels}"
    )
else:
    print("Signal-level EEG plotting skipped because the current .set/.fdt files are annex pointers rather than real EEGLAB binaries.")
    print("After retrieving the raw EEG files, rerun this cell to generate participant-average EEG plots across channels.")


Identifying common channels ...
Identifying common channels ...


,participant_id,n_trials_total,n_familiar_epochs,n_not_familiar_epochs,familiar_fraction
0,sub-01,74,40,34,0.541
1,sub-02,75,47,28,0.627
2,sub-03,101,87,14,0.861
3,sub-04,78,25,53,0.321
4,sub-05,66,47,19,0.712


,count,mean,std,min,25%,50%,75%,max
n_trials_total,29.0,71.52,11.24,51.0,63.00,69.00,77.00,101.00
n_familiar_epochs,29.0,35.52,20.08,0.0,25.00,34.00,44.00,90.00
n_not_familiar_epochs,29.0,36.00,16.03,9.0,27.00,32.00,47.00,69.00
familiar_fraction,29.0,0.49,0.22,0.0,0.33,0.52,0.63,0.91


Rendered participant-average EEG plots for 29 participants across 32 shared channels. Displayed channel traces: ['Cz', 'Pz', 'Oz']


### EEG Coverage Interpretation

All `29` participants expose EEG metadata with `32` channels sampled at `1000 Hz`, which gives the branch a consistent recording structure for feature extraction. When the real EEGLAB binaries are present locally, the final plots let you inspect condition-averaged signal patterns; when they are missing or annex placeholders, the metadata tables still confirm that the dataset is organized cleanly enough for reproducible preprocessing.
